<a href="https://colab.research.google.com/github/rodriguesrleonardo/rodriguesrleonardo/blob/main/Extra%C3%A7%C3%A3o_de_dados_SPED_ICMS_IPI_Vers%C3%A3o_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Obs.: Precisa alimentar a pasta SPED. Ideal sempre apagar tudo e criar uma pasta nova com o mesmo nome SPED e fazer upload dos arquivos de um determinado período

# 1º Bibliotecas necessárias

In [2]:
!pip install pandas openpyxl xlsxwriter -q

# 2º Importações e configurações

In [30]:
import os
import pandas as pd
from datetime import datetime

PASTA_SPED = "/content/SPED"

from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

ARQUIVO_EXCEL = f"/content/SPED/RESULTADOS/Consolidado_SPED_{timestamp}.xlsx"
ARQUIVO_CFOP = f"/content/SPED/RESULTADOS/CFOP_DETALHADO_{timestamp}.csv"

# 3º Funções Auxiliares

In [4]:
def to_float(valor):
    if valor is None:
        return 0.0

    valor = str(valor).strip()

    if valor == "":
        return 0.0

    try:
        return float(valor.replace(".", "").replace(",", "."))
    except:
        return 0.0


def obter_raiz_cnpj(cnpj):
    return str(cnpj)[:8]


def obter_periodo(data_ini):
    if len(data_ini) != 8:
        return ""

    return f"{data_ini[4:8]}-{data_ini[2:4]}"


def tipo_cfop(cfop):
    cfop = str(cfop)

    if cfop.startswith("1"):
        return "Entrada Estadual"

    if cfop.startswith("2"):
        return "Entrada Interestadual"

    if cfop.startswith("3"):
        return "Importação"

    if cfop.startswith("5"):
        return "Saída Estadual"

    if cfop.startswith("6"):
        return "Saída Interestadual"

    if cfop.startswith("7"):
        return "Exportação"

    return "Não Identificado"

# 4º Estrutura de dados

In [5]:
empresas = []
cfop_detalhado = []
apuracoes = []
ajustes = []
obrigacoes = []

# 5º Leitura dos arquivos SPED

In [6]:
arquivos_txt = [
    arq
    for arq in os.listdir(PASTA_SPED)
    if arq.lower().endswith(".txt")
]

print(f"{len(arquivos_txt)} arquivos encontrados.")

187 arquivos encontrados.


# 6º Processamento de dados

In [7]:
for arquivo in arquivos_txt:

    caminho = os.path.join(PASTA_SPED, arquivo)

    print(f"Processando {arquivo}")

    empresa = {
        "RAIZ_CNPJ": "",
        "CNPJ": "",
        "IE": "",
        "RAZAO_SOCIAL": "",
        "UF": "",
        "PERIODO": "",
        "ARQUIVO": arquivo
    }

    ultimo_num_doc = ""
    ultima_chave = ""

    with open(caminho, "r", encoding="latin-1") as f:

        for linha in f:

            linha = linha.strip()

            if not linha:
                continue

            campos = linha.split("|")

            if len(campos) < 2:
                continue

            registro = campos[1]

            # -----------------------------------
            # REGISTRO 0000
            # -----------------------------------

            if registro == "0000":

                data_ini = campos[4]
                data_fim = campos[5]

                empresa["CNPJ"] = campos[7]
                empresa["RAIZ_CNPJ"] = obter_raiz_cnpj(campos[7])
                empresa["RAZAO_SOCIAL"] = campos[6]
                empresa["UF"] = campos[9]
                empresa["IE"] = campos[10]
                empresa["PERIODO"] = obter_periodo(data_ini)

                empresas.append({
                    **empresa,
                    "DATA_INICIAL": data_ini,
                    "DATA_FINAL": data_fim
                })

            # -----------------------------------
            # C100
            # -----------------------------------

            elif registro == "C100":

                ultimo_num_doc = campos[8] if len(campos) > 8 else ""

                ultima_chave = campos[9] if len(campos) > 9 else ""

            # -----------------------------------
            # C190
            # -----------------------------------

            elif registro == "C190":

                cfop_detalhado.append({

                    **empresa,

                    "TIPO_OPERACAO": tipo_cfop(campos[3]),

                    "NUM_DOC": ultimo_num_doc,
                    "CHV_NFE": ultima_chave,

                    "CST": campos[2],
                    "CFOP": campos[3],
                    "ALIQ_ICMS": to_float(campos[4]),

                    "VL_OPR": to_float(campos[5]),
                    "VL_BC_ICMS": to_float(campos[6]),
                    "VL_ICMS": to_float(campos[7]),
                    "VL_BC_ST": to_float(campos[8]),
                    "VL_ICMS_ST": to_float(campos[9]),
                    "VL_RED_BC": to_float(campos[10])
                })

            # -----------------------------------
            # E110
            # -----------------------------------

            elif registro == "E110":

                apuracoes.append({

                    **empresa,

                    "VL_TOT_DEBITOS": to_float(campos[2]),
                    "VL_AJ_DEBITOS": to_float(campos[3]),
                    "VL_TOT_AJ_DEBITOS": to_float(campos[4]),
                    "VL_ESTORNOS_CRED": to_float(campos[5]),
                    "VL_TOT_CREDITOS": to_float(campos[6]),
                    "VL_AJ_CREDITOS": to_float(campos[7]),
                    "VL_TOT_AJ_CREDITOS": to_float(campos[8]),
                    "VL_ESTORNOS_DEB": to_float(campos[9]),
                    "VL_SLD_CREDOR_ANT": to_float(campos[10]),
                    "VL_SLD_APURADO": to_float(campos[11]),
                    "VL_TOT_DED": to_float(campos[12]),
                    "VL_ICMS_RECOLHER": to_float(campos[13]),
                    "VL_SLD_CREDOR_TRANSPORTAR": to_float(campos[14])
                })

            # -----------------------------------
            # E111
            # -----------------------------------

            elif registro == "E111":

                ajustes.append({

                    **empresa,

                    "COD_AJUSTE": campos[2],
                    "DESCRICAO": campos[3],
                    "VALOR": to_float(campos[4])
                })

            # -----------------------------------
            # E116
            # -----------------------------------

            elif registro == "E116":

                obrigacoes.append({

                    **empresa,

                    "COD_ORIGEM": campos[2],
                    "VALOR": to_float(campos[3]),
                    "VENCIMENTO": campos[4],
                    "COD_RECEITA": campos[5],
                    "DESCRICAO": campos[10] if len(campos) > 10 else ""
                })

Processando 17625216001036-796186792116-20260401-20260430-0-983574799314D4D91B0BE903E518C71671AE26C7-SPED-EFD.txt
Processando 17625216002512-796248079116-20260401-20260430-0-D592DE4CF055225455AB1A106206D1E3F9265673-SPED-EFD.txt
Processando 17625216000307-796186844116-20260401-20260430-0-13BAD98D76303B4C652312C365981784675F8AAF-SPED-EFD.txt
Processando 27197888015930-795598280110-20260401-20260430-0-CD20A726FDCA96E7DFEE50EB8E2075BD8F190085-SPED-EFD.txt
Processando 27197888027270-054691966-20260401-20260430-0-A86D28E342780FAF0CC5EDA380B9F04CA3549548-SPED-EFD.txt
Processando 17625216000226-796186817113-20260401-20260430-0-EE105671F46070CB1DA35E45A454E95951CC385A-SPED-EFD.txt
Processando 27197888018955-0628832431288-20260401-20260430-0-353E76302AF060F2E6E370CA8938145E1625CBFC-SPED-EFD.txt
Processando 27197888022472-795884548110-20260401-20260430-0-3C197BEF95BE4D9909A81E47AA8448B23E3A9BEC-SPED-EFD.txt
Processando 27197888017207-0738333402180-20260401-20260430-0-3D539483A2C6C04252A58A232967A

# Verificação rápida

In [8]:
print("Empresas:", len(empresas))
print("CFOP Detalhado:", len(cfop_detalhado))
print("Apurações:", len(apuracoes))
print("Ajustes:", len(ajustes))
print("Obrigações:", len(obrigacoes))

Empresas: 187
CFOP Detalhado: 914671
Apurações: 187
Ajustes: 258
Obrigações: 138


# Conferência

In [9]:
pd.DataFrame(cfop_detalhado).head()

,RAIZ_CNPJ,CNPJ,IE,RAZAO_SOCIAL,UF,PERIODO,ARQUIVO,TIPO_OPERACAO,NUM_DOC,CHV_NFE,CST,CFOP,ALIQ_ICMS,VL_OPR,VL_BC_ICMS,VL_ICMS,VL_BC_ST,VL_ICMS_ST,VL_RED_BC
0,17625216,17625216001036,796186792116,DUFRY LOJAS FRANCAS LTDA,SP,2026-04,17625216001036-796186792116-20260401-20260430-...,Entrada Estadual,000013383,35260417625216000145550500000133831938850002,140,1152,0.0,24019.35,0.0,0.0,0.0,0.0,0.0
1,17625216,17625216001036,796186792116,DUFRY LOJAS FRANCAS LTDA,SP,2026-04,17625216001036-796186792116-20260401-20260430-...,Entrada Estadual,000013376,35260417625216000145550500000133761124554482,140,1152,0.0,230.67,0.0,0.0,0.0,0.0,0.0
2,17625216,17625216001036,796186792116,DUFRY LOJAS FRANCAS LTDA,SP,2026-04,17625216001036-796186792116-20260401-20260430-...,Entrada Estadual,000013389,35260417625216000145550500000133891568225293,140,1152,0.0,163.44,0.0,0.0,0.0,0.0,0.0
3,17625216,17625216001036,796186792116,DUFRY LOJAS FRANCAS LTDA,SP,2026-04,17625216001036-796186792116-20260401-20260430-...,Entrada Estadual,000013413,35260417625216000145550500000134131812939099,140,1152,0.0,301.86,0.0,0.0,0.0,0.0,0.0
4,17625216,17625216001036,796186792116,DUFRY LOJAS FRANCAS LTDA,SP,2026-04,17625216001036-796186792116-20260401-20260430-...,Entrada Estadual,000013450,35260417625216000145550500000134501844385475,140,1152,0.0,23417.57,0.0,0.0,0.0,0.0,0.0


# 7º Criação dos Dataframes

In [11]:
df_empresas = pd.DataFrame(empresas)
df_cfop = pd.DataFrame(cfop_detalhado)
df_apuracao = pd.DataFrame(apuracoes)
df_ajustes = pd.DataFrame(ajustes)
df_obrigacoes = pd.DataFrame(obrigacoes)

print("Empresas:", len(df_empresas))
print("CFOP:", len(df_cfop))
print("Apuração:", len(df_apuracao))
print("Ajustes:", len(df_ajustes))
print("Obrigações:", len(df_obrigacoes))

Empresas: 187
CFOP: 914671
Apuração: 187
Ajustes: 258
Obrigações: 138


# Teste

In [12]:
df_cfop.head()

,RAIZ_CNPJ,CNPJ,IE,RAZAO_SOCIAL,UF,PERIODO,ARQUIVO,TIPO_OPERACAO,NUM_DOC,CHV_NFE,CST,CFOP,ALIQ_ICMS,VL_OPR,VL_BC_ICMS,VL_ICMS,VL_BC_ST,VL_ICMS_ST,VL_RED_BC
0,17625216,17625216001036,796186792116,DUFRY LOJAS FRANCAS LTDA,SP,2026-04,17625216001036-796186792116-20260401-20260430-...,Entrada Estadual,000013383,35260417625216000145550500000133831938850002,140,1152,0.0,24019.35,0.0,0.0,0.0,0.0,0.0
1,17625216,17625216001036,796186792116,DUFRY LOJAS FRANCAS LTDA,SP,2026-04,17625216001036-796186792116-20260401-20260430-...,Entrada Estadual,000013376,35260417625216000145550500000133761124554482,140,1152,0.0,230.67,0.0,0.0,0.0,0.0,0.0
2,17625216,17625216001036,796186792116,DUFRY LOJAS FRANCAS LTDA,SP,2026-04,17625216001036-796186792116-20260401-20260430-...,Entrada Estadual,000013389,35260417625216000145550500000133891568225293,140,1152,0.0,163.44,0.0,0.0,0.0,0.0,0.0
3,17625216,17625216001036,796186792116,DUFRY LOJAS FRANCAS LTDA,SP,2026-04,17625216001036-796186792116-20260401-20260430-...,Entrada Estadual,000013413,35260417625216000145550500000134131812939099,140,1152,0.0,301.86,0.0,0.0,0.0,0.0,0.0
4,17625216,17625216001036,796186792116,DUFRY LOJAS FRANCAS LTDA,SP,2026-04,17625216001036-796186792116-20260401-20260430-...,Entrada Estadual,000013450,35260417625216000145550500000134501844385475,140,1152,0.0,23417.57,0.0,0.0,0.0,0.0,0.0


# 8º CFOP_RESUMIDO

In [13]:
df_cfop_resumido = (
    df_cfop
    .groupby(
        [
            "RAIZ_CNPJ",
            "CNPJ",
            "IE",
            "RAZAO_SOCIAL",
            "UF",
            "PERIODO",
            "CFOP",
            "CST",
            "ALIQ_ICMS"
        ],
        as_index=False
    )
    .agg(
        {
            "VL_OPR":"sum",
            "VL_BC_ICMS":"sum",
            "VL_ICMS":"sum",
            "VL_BC_ST":"sum",
            "VL_ICMS_ST":"sum",
            "VL_RED_BC":"sum"
        }
    )
)

# 9º RESUMO_EXECUTIVO

In [37]:
entradas = (
    df_cfop[
        df_cfop["CFOP"].astype(str).str.startswith(("1","2","3"))
    ]
    .groupby(["CNPJ","PERIODO"])["VL_OPR"]
    .sum()
    .reset_index()
    .rename(columns={"VL_OPR":"TOTAL_ENTRADAS"})
)

saidas = (
    df_cfop[
        df_cfop["CFOP"].astype(str).str.startswith(("5","6","7"))
    ]
    .groupby(["CNPJ","PERIODO"])["VL_OPR"]
    .sum()
    .reset_index()
    .rename(columns={"VL_OPR":"TOTAL_SAIDAS"})
)

resumo = df_empresas[
    [
        "RAIZ_CNPJ",
        "CNPJ",
        "IE",
        "RAZAO_SOCIAL",
        "UF",
        "PERIODO",
        "ARQUIVO"
    ]
].copy()

resumo = resumo.merge(
    entradas,
    on=["CNPJ","PERIODO"],
    how="left"
)

resumo = resumo.merge(
    saidas,
    on=["CNPJ","PERIODO"],
    how="left"
)

resumo["TOTAL_ENTRADAS"] = resumo["TOTAL_ENTRADAS"].fillna(0)
resumo["TOTAL_SAIDAS"] = resumo["TOTAL_SAIDAS"].fillna(0)

# 10º ICMS RECOLHER E FECP

In [38]:
icms_recolher = (
    df_apuracao[
        ["CNPJ","PERIODO","VL_ICMS_RECOLHER"]
    ]
)

resumo = resumo.merge(
    icms_recolher,
    on=["CNPJ","PERIODO"],
    how="left"
)

resumo["VL_ICMS_RECOLHER"] = resumo["VL_ICMS_RECOLHER"].fillna(0)

# ==========================================
# FECP (RJ)
# Utiliza apenas o ajuste RJ040010 para evitar duplicidade
# ==========================================

fecp = (
    df_ajustes[
        df_ajustes["COD_AJUSTE"] == "RJ040010"
    ]
    .groupby(
        ["CNPJ","PERIODO"]
    )["VALOR"]
    .sum()
    .reset_index()
    .rename(columns={"VALOR":"FECP"})
)

resumo = resumo.merge(
    fecp,
    on=["CNPJ","PERIODO"],
    how="left"
)

if "FECP" not in resumo.columns:
    resumo["FECP"] = 0

resumo["FECP"] = resumo["FECP"].fillna(0)

# 11º Exportar CSV Detalhado

In [39]:
df_cfop.to_csv(
    ARQUIVO_CFOP,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print("CSV gerado.")

CSV gerado.


# 12º Exportar Excel

In [40]:
with pd.ExcelWriter(
    ARQUIVO_EXCEL,
    engine="xlsxwriter"
) as writer:

    df_empresas.to_excel(
        writer,
        sheet_name="EMPRESAS",
        index=False
    )

    df_cfop_resumido.to_excel(
        writer,
        sheet_name="CFOP_RESUMIDO",
        index=False
    )

    df_apuracao.to_excel(
        writer,
        sheet_name="APURACAO_ICMS",
        index=False
    )

    df_ajustes.to_excel(
        writer,
        sheet_name="AJUSTES_E111",
        index=False
    )

    df_obrigacoes.to_excel(
        writer,
        sheet_name="OBRIGACOES_E116",
        index=False
    )

    resumo.to_excel(
        writer,
        sheet_name="RESUMO_EXECUTIVO",
        index=False
    )

print("Excel gerado com sucesso.")
print(ARQUIVO_EXCEL)
print(ARQUIVO_CFOP)

Excel gerado com sucesso.
/content/SPED/RESULTADOS/Consolidado_SPED_20260609_162857.xlsx
/content/SPED/RESULTADOS/CFOP_DETALHADO_20260609_162857.csv
